In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats.mstats import winsorize
from scipy.stats import skew
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from feature_engine.selection import RecursiveFeatureAddition
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
import shap

I am utilizing this code as guidance, https://github.com/inab-certh/Predicting-COVID-19-severity-through-interpretable-AI-analysis-of-plasma-proteomics/blob/main/Task%202.ipynb

## Reading in ISB and ICL Data

In [ ]:
isb_clinical = pd.read_excel('final_data/ISB/isb_clinical.xlsx')
isb_proteomics = pd.read_excel('final_data/ISB/isb_proteomics.xlsx')
isb_metabolomics = pd.read_excel('final_data/ISB/isb_metabolomics.xlsx')

In [ ]:
isb_clinical.columns

**Removing Ethnicity**

In [ ]:
isb_clinical.drop('ethnicity', axis=1, inplace=True)

Matching **sample ids** for ICL Wave 1

In [ ]:
isb_proteomics.set_index('subject_id', inplace=True)
isb_metabolomics.set_index('subject_id', inplace=True)
isb_clinical.set_index('subject_id', inplace=True)

## Checking for Missing Values

In [ ]:
pro_rows_with_na = isb_proteomics.isna().any(axis=1)

print("Number of rows with at least one missing value:")
print("ISB Proteomics: ", pro_rows_with_na.sum())

In [ ]:
met_rows_with_na = isb_metabolomics.isna().any(axis=1)

print("Number of rows with at least one missing value:")
print("ISB Metabolomics: ", met_rows_with_na.sum())

In [ ]:
relevant_clinical_columns = ['who_severity', 'sex', 'age', 'cigarette_smoking', 'kidney_disease', 
                            'chronic_hypertension', 'cancer', 'asthma', 'copd', 'coronary_artery_disease', 
                            'respiratory_support']

isb_clinical = isb_clinical[relevant_clinical_columns]
isb_clinical

In [ ]:
clin_rows_with_na = isb_clinical.isna().any(axis=1)

print("Number of rows with at least one missing value:")
print("ISB Clinical: ", clin_rows_with_na.sum())

## Binning Age

In [ ]:
isb_clinical['age']

In [ ]:
isb_clinical['age'].describe()

Categorizing ages as follows:
- ages 0 to 40
- ages 41 to 60
- ages 61 to 80
- ages 81 and above

In [ ]:
age_bins = [0, 40, 60, 80, 100]
age_labels = [0, 1, 2, 3] 

isb_clinical['age'] = pd.cut(isb_clinical['age'], bins=age_bins, labels=age_labels, right=True).astype(int)

print("ISB Data:")
print(isb_clinical['age'].value_counts())

## Modifying Target

In [ ]:
def target_recode_scale(value):
    if value in [1, 2, 3, '1 or 2']:
        return 'mild'
    elif value in [4, 5]:
        return 'moderate'
    elif value in [6, 7]:
        return 'severe'

isb_clinical['who_severity'] = isb_clinical['who_severity'].apply(target_recode_scale)

In [ ]:
isb_clinical['who_severity'].value_counts()

## Splitting Data

In [ ]:
y = isb_clinical['who_severity']

label_order = {'mild': 0, 'moderate': 1, 'severe': 2}

y = y.map(label_order)

print("Target:")
y.value_counts()

In [ ]:
y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

isb_proteomics_mod = isb_proteomics.drop(['Blood Draw', 'Healthy or INCOV', 'age', 'sex', 'BMI'], axis = 1)
isb_metabolomics_mod = isb_metabolomics.drop(['Blood Draw', 'Healthy or INCOV', 'age', 'sex', 'BMI'], axis = 1)
isb_clinical_mod = isb_clinical.drop(['who_severity'], axis = 1)

X_metabolomic_temp, X_metabolomic_test, y_metabolomic_temp, y_metabolomic_test = train_test_split(isb_metabolomics_mod, y, test_size=0.2, random_state=42, stratify=y)
X_proteomic_temp, X_proteomic_test, y_proteomic_temp, y_proteomic_test = train_test_split(isb_proteomics_mod, y, test_size=0.2, random_state=42, stratify=y)
X_clinical_temp, X_clinical_test, y_clinical_temp, y_clinical_test = train_test_split(isb_clinical_mod, y, test_size=0.2, random_state=42, stratify=y)

X_metabolomic_train, X_metabolomic_val, y_metabolomic_train, y_metabolomic_val = train_test_split(X_metabolomic_temp, y_metabolomic_temp, test_size=0.5, random_state=42, stratify=y_metabolomic_temp)
X_proteomic_train, X_proteomic_val, y_proteomic_train, y_proteomic_val = train_test_split(X_proteomic_temp, y_proteomic_temp, test_size=0.5, random_state=42, stratify=y_proteomic_temp)
X_clinical_train, X_clinical_val, y_clinical_train, y_clinical_val = train_test_split(X_clinical_temp, y_clinical_temp, test_size=0.5, random_state=42, stratify=y_clinical_temp)

In [ ]:
print("Proteomics Train Target:", y_proteomic_train.value_counts(normalize=True))  

print("Proteomics Test Target:", y_proteomic_test.value_counts(normalize=True))

print("Proteomics Val Target:",y_proteomic_val.value_counts(normalize=True))  

In [ ]:
X_clinical_train = X_clinical_train.copy()
X_clinical_test = X_clinical_test.copy()
X_clinical_val = X_clinical_val.copy()

## Encoding Clinical Features

Addressing the unknown value in some clinical features

In [ ]:
def replace_unknown_with_mode(train_column, test_column):
    mode_value = train_column.mode()[0]
    return train_column.replace('Unknown', mode_value), test_column.replace('Unknown', mode_value)

In [ ]:
isb_clinical['cancer'].value_counts()

In [ ]:
X_clinical_train['cancer'], X_clinical_test['cancer'] = replace_unknown_with_mode(X_clinical_train['cancer'], X_clinical_test['cancer'])
X_clinical_train['cancer'], X_clinical_val['cancer'] = replace_unknown_with_mode(X_clinical_train['cancer'], X_clinical_val['cancer'])

In [ ]:
X_clinical_train['cigarette_smoking'], X_clinical_test['cigarette_smoking'] = replace_unknown_with_mode(X_clinical_train['cigarette_smoking'], X_clinical_test['cigarette_smoking'])
X_clinical_train['cigarette_smoking'], X_clinical_val['cigarette_smoking'] = replace_unknown_with_mode(X_clinical_train['cigarette_smoking'], X_clinical_val['cigarette_smoking'])

In [ ]:
X_clinical_train['asthma'], X_clinical_test['asthma'] = replace_unknown_with_mode(X_clinical_train['asthma'], X_clinical_test['asthma'])
X_clinical_train['asthma'], X_clinical_val['asthma'] = replace_unknown_with_mode(X_clinical_train['asthma'], X_clinical_val['asthma'])

In [ ]:
X_clinical_train['copd'], X_clinical_test['copd'] = replace_unknown_with_mode(X_clinical_train['copd'], X_clinical_test['copd'])
X_clinical_train['copd'], X_clinical_val['copd'] = replace_unknown_with_mode(X_clinical_train['copd'], X_clinical_val['copd'])

In [ ]:
X_clinical_train['coronary_artery_disease'], X_clinical_test['coronary_artery_disease'] = replace_unknown_with_mode(X_clinical_train['coronary_artery_disease'], X_clinical_test['coronary_artery_disease'])
X_clinical_train['coronary_artery_disease'], X_clinical_val['coronary_artery_disease'] = replace_unknown_with_mode(X_clinical_train['coronary_artery_disease'], X_clinical_val['coronary_artery_disease'])

In [ ]:
cigarette_ordering = [['Never', 'Former', 'Current']]
respiratory_support_ordering = [['None', 'Other', 'Nasal cannula', 'High flow nasal cannula (HFNC)']]

In [ ]:
from pandas.api.types import CategoricalDtype
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

encoder = OrdinalEncoder(categories=cigarette_ordering)

X_clinical_train['cigarette_smoking'] = encoder.fit_transform(X_clinical_train[['cigarette_smoking']]).astype(int)

X_clinical_val['cigarette_smoking'] = encoder.transform(X_clinical_val[['cigarette_smoking']]).astype(int)
X_clinical_test['cigarette_smoking'] = encoder.transform(X_clinical_test[['cigarette_smoking']]).astype(int)

encoder_respiratory_support = OrdinalEncoder(categories=respiratory_support_ordering)

X_clinical_train['respiratory_support'] = encoder_respiratory_support.fit_transform(X_clinical_train[['respiratory_support']]).astype(int)
X_clinical_val['respiratory_support'] = encoder_respiratory_support.transform(X_clinical_val[['respiratory_support']]).astype(int)
X_clinical_test['respiratory_support'] = encoder_respiratory_support.transform(X_clinical_test[['respiratory_support']]).astype(int)

age_train = X_clinical_train.pop('age')
age_val = X_clinical_val.pop('age')
age_test = X_clinical_test.pop('age')

binary_columns = ['sex', 'kidney_disease', 'chronic_hypertension', 'cancer', 'asthma', 'copd', 'coronary_artery_disease']
binary_mapping = {'Male': 1, 'Yes': 1, 'Female': 0, 'No': 0}

for column in binary_columns:
    X_clinical_train[column] = X_clinical_train[column].map(binary_mapping)
    X_clinical_val[column] = X_clinical_val[column].map(binary_mapping)
    X_clinical_test[column] = X_clinical_test[column].map(binary_mapping)

X_clinical_train['age'] = age_train


X_clinical_val['age'] = age_val


X_clinical_test['age'] = age_test

In [ ]:
X_clinical_train

## Checking Correlation and Multicollinearity

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = X_clinical_train.corr().abs()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, linewidths=1, annot=False, fmt=".2f")
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

df_const = add_constant(X_clinical_test)

vif_data = pd.DataFrame()
vif_data["Variable"] = df_const.columns
vif_data["VIF"] = [variance_inflation_factor(df_const.values, i) for i in range(df_const.shape[1])]

print(vif_data)

## Omics Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

proteomics_scaling_selection = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('selector', RFE(estimator=LogisticRegression(), n_features_to_select=100))
])

metabolomics_scaling_selection = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('selector', RFE(estimator=LogisticRegression(), n_features_to_select=100))
])

### Proteomics

In [ ]:
X_proteomic_train_preprocessed = proteomics_scaling_selection.fit_transform(X_proteomic_train, y_proteomic_train)

X_proteomic_val_preprocessed = proteomics_scaling_selection.transform(X_proteomic_val)
X_proteomic_test_preprocessed = proteomics_scaling_selection.transform(X_proteomic_test)

selected_features_proteomics = X_proteomic_train.columns[proteomics_scaling_selection.named_steps['selector'].support_]

X_proteomic_train_preprocessed = pd.DataFrame(X_proteomic_train_preprocessed, columns=selected_features_proteomics, index=X_proteomic_train.index)
X_proteomic_val_preprocessed = pd.DataFrame(X_proteomic_val_preprocessed, columns=selected_features_proteomics, index=X_proteomic_val.index)
X_proteomic_test_preprocessed = pd.DataFrame(X_proteomic_test_preprocessed, columns=selected_features_proteomics, index=X_proteomic_test.index)

### Metabolomics

In [ ]:
X_metabolomic_train_preprocessed = metabolomics_scaling_selection.fit_transform(X_metabolomic_train, y_metabolomic_train)
X_metabolomic_val_preprocessed = metabolomics_scaling_selection.transform(X_metabolomic_val)
X_metabolomic_test_preprocessed = metabolomics_scaling_selection.transform(X_metabolomic_test)

selected_features_metabolomics = X_metabolomic_train.columns[metabolomics_scaling_selection.named_steps['selector'].support_]

X_metabolomic_train_preprocessed = pd.DataFrame(X_metabolomic_train_preprocessed, columns=selected_features_metabolomics, index=X_metabolomic_train.index)
X_metabolomic_val_preprocessed = pd.DataFrame(X_metabolomic_val_preprocessed, columns=selected_features_metabolomics, index=X_metabolomic_val.index)
X_metabolomic_test_preprocessed = pd.DataFrame(X_metabolomic_test_preprocessed, columns=selected_features_metabolomics, index=X_metabolomic_test.index)

## PCA

In [ ]:
original_columns_proteomics = X_proteomic_train_preprocessed.columns.tolist()
original_columns_metabolomics = X_metabolomic_train_preprocessed.columns.tolist()

In [ ]:
pca_proteomics = PCA(.90)
pca_metabolomics = PCA(.90)

### Transforming Proteomics Data

In [ ]:
X_proteomic_train_preprocessed = pd.DataFrame(pca_proteomics.fit_transform(X_proteomic_train_preprocessed), index=X_proteomic_train_preprocessed.index)
X_proteomic_test_preprocessed = pd.DataFrame(pca_proteomics.transform(X_proteomic_test_preprocessed), index=X_proteomic_test_preprocessed.index)
X_proteomic_val_preprocessed = pd.DataFrame(pca_proteomics.transform(X_proteomic_val_preprocessed), index=X_proteomic_val_preprocessed.index)

loadings_proteomics = pca_proteomics.components_.T * np.sqrt(pca_proteomics.explained_variance_)

loading_matrix_proteomics= pd.DataFrame(loadings_proteomics, index=original_columns_proteomics)
loading_matrix_proteomics

In [ ]:
load_trans_proteomics = loading_matrix_proteomics.T
loading_pow_proteomics = {}


for i in load_trans_proteomics.columns:
    magnitude = load_trans_proteomics[i].pow(2).sum()
    loading_pow_proteomics[i] = magnitude

pd.DataFrame.from_dict(loading_pow_proteomics, orient='index').to_excel('loadingspowProteomics.xlsx')

### Transforming Metabolomics Data

In [ ]:
X_metabolomic_train_preprocessed = pd.DataFrame(pca_metabolomics.fit_transform(X_metabolomic_train_preprocessed), index=X_metabolomic_train_preprocessed.index)
X_metabolomic_test_preprocessed = pd.DataFrame(pca_metabolomics.transform(X_metabolomic_test_preprocessed), index=X_metabolomic_test_preprocessed.index)
X_metabolomic_val_preprocessed = pd.DataFrame(pca_metabolomics.transform(X_metabolomic_val_preprocessed), index=X_metabolomic_val_preprocessed.index)

loadings_metabolomics = pca_metabolomics.components_.T * np.sqrt(pca_metabolomics.explained_variance_)

loading_matrix_metabolomics = pd.DataFrame(loadings_metabolomics, index=original_columns_metabolomics)
loading_matrix_metabolomics

In [ ]:
load_trans_metabolomics = loading_matrix_metabolomics.T
loading_pow_metabolomics = {}


for i in load_trans_metabolomics.columns:
    magnitude = load_trans_metabolomics[i].pow(2).sum()
    loading_pow_metabolomics[i] = magnitude

pd.DataFrame.from_dict(loading_pow_metabolomics, orient='index').to_excel('loadingspowMetabolomics.xlsx')

## Model Training

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV
from numpy import mean
from sklearn.metrics import make_scorer, accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

models = {
    'logistic_regression': LogisticRegression(),
    'decision_tree': DecisionTreeClassifier(),
    'rand_forest': RandomForestClassifier(),
    'extra_trees': ExtraTreesClassifier(),
    'mlp': MLPClassifier(),
    'svm': SVC(),
    'ada_boost': AdaBoostClassifier(),
    'xgb': xgb.XGBClassifier(),
    'gbm': GradientBoostingClassifier(),
    'qda': QuadraticDiscriminantAnalysis(),
}

log_params = {
    'logistic_regression__penalty': ['l1', 'l2'],
    'logistic_regression__C': [0.1, 1, 10],
    'logistic_regression__solver': ['liblinear', 'saga'],
    'logistic_regression__max_iter': [100, 200, 300, 400, 500, 600, 700, 800, 900],
    'logistic_regression__class_weight': [None, 'balanced']
}


dt_params = {
    'decision_tree__criterion': ['gini', 'entropy'],
    'decision_tree__max_depth': [None,1, 5, 10, 50, 100],
    'decision_tree__min_samples_split': [2, 5, 10],
    'decision_tree__min_samples_leaf': [1, 2, 4],
    'decision_tree__splitter':['best','random'],
    'decision_tree__class_weight': [None, 'balanced']
}

rf_params = {
    'rand_forest__n_estimators':[50,100,500,1000,2000],
    'rand_forest__criterion': ['gini', 'entropy'],
    'rand_forest__max_depth': [None, 5, 10, 20],
    'rand_forest__min_samples_split': [2, 5, 10],
    'rand_forest__min_samples_leaf': [1, 2, 4],
    'rand_forest__class_weight': [None, 'balanced']
}

et_params = {
    'extra_trees__n_estimators': [100, 200, 500, 1000, 2000],
    'extra_trees__criterion': ['gini', 'entropy'],
    'extra_trees__max_depth': [None, 5, 10],
    'extra_trees__min_samples_split': [2, 5, 10],
    'extra_trees__min_samples_leaf': [1, 2, 4],
    'extra_trees__class_weight': [None, 'balanced']
}

mlp_params = {
    'mlp__hidden_layer_sizes': [(100,), (100, 50), (50, 50), (50,50,50)],
    'mlp__activation': ['relu', 'tanh', 'logistic', 'identity'],
    'mlp__solver': ['adam', 'sgd', 'lbfgs'],
    'mlp__learning_rate':['constant','adaptive'],
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.05],
    'mlp__max_iter': [100, 200, 300, 400, 500, 600, 700, 800, 900]
}

svm_params = {
    'svm__C':[0.1, 1, 2, 5, 10, 50, 100, 500],
    'svm__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'svm__gamma': ['scale', 'auto'],
    'svm__probability': [True],
    'svm__class_weight': [None, 'balanced'],
    'svm__max_iter': [100, 200, 300, 400, 500, 600, 700, 800, 900]
}

ada_params = {
    'ada_boost__n_estimators': [10, 20, 30, 50, 100, 200, 500, 1000],
    'ada_boost__learning_rate': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}

xgb_params = {
    'xgb__n_estimators': [100, 200, 500],
    'xgb__learning_rate': [0.01, 0.03, 0.06, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7],
    'xgb__max_depth': [3, 5, 7, 9, 11],
    'xgb__subsample': [0.8, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0],
    'xgb__gamma': [0, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.4, 12.8, 25.6, 51.2, 102.4, 200],
    'xgb__reg_alpha': [0, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.4, 12.8, 25.6, 51.2, 102.4, 200],
    'xgb__reg_lambda': [0, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2, 6.4, 12.8, 25.6, 51.2, 102.4, 200]
}

gbm_params = {
    'gbm__n_estimators': [50, 100, 200, 500, 1000, 2000],
    'gbm__learning_rate': [0.01, 0.1, 0.2, 0.3, 0.4],
    'gbm__max_depth': [3, 5, 7],
    'gbm__subsample': [0.8, 1.0],
    'gbm__max_features': ['sqrt', 'log2']
}

qda_params = {
    'qda__reg_param': [0, 0.5, 1]  
}



params = {
    'logistic_regression': log_params,
    'decision_tree': dt_params,
    'rand_forest': rf_params,
    'extra_trees': et_params,
    'mlp': mlp_params,
    'svm': svm_params,
    'ada_boost': ada_params,
    'xgb': xgb_params,
    'gbm': gbm_params,
    'qda': qda_params,
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import BorderlineSMOTE, SVMSMOTE, RandomOverSampler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report
import time
import matplotlib.pyplot as plt


def model_search(models, params, over, under, X_train, y_train, X_test, y_test, X_val, y_val):
    print(f"Starting model search at: {time.ctime()}")
    print(f"Number of models: {len(models)}")
    training_times = []
    max_score = 0
    max_model = None
    max_model_params = None
    estimators_modelsearch = pd.DataFrame()
    models_est = []
    parameters = []
    lscore = []
    lroc = []
    lfscore_macro = []
    lfscore_weighted = []
    lprecision = []
    lrecall = []
    lrecall_severe = []
    lclass_report = []
    lcvscore = []
    lscore_val = []
    lroc_val = []
    lfscore_macro_val = []
    lfscore_weighted_val = []
    lprecision_val = []
    lrecall_val = []
    lrecall_severe_val = []
    lclass_report_val = []

    scoring = {
        'balanced_accuracy': 'balanced_accuracy',
        'accuracy': 'accuracy',
        'roc_auc': 'roc_auc_ovr',
        'precision': 'precision_macro',
        'recall_macro': make_scorer(recall_score, average='macro'),
        'f1_macro': make_scorer(f1_score, average='macro'),
        'f1_weighted': make_scorer(f1_score, average='weighted')
    }

    cv = StratifiedKFold(10)
    
    processing_times.clear()

    for i, j in models.items():
        start_time = time.time()
        processing_times = None  

        try:
            pipeline = Pipeline([('over', over), ('under', under), (i, j)])

            rs = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=params[i],
                scoring=scoring,
                refit='roc_auc',
                cv=cv,
                n_iter=10,
                random_state=42,
                n_jobs=-1
            )

            rs.fit(X_train, y_train)

        except Exception as e:
            print(f"Error while fitting model {i}: {e}")
            processing_times.append(None)  
            continue



        y_pred = rs.predict(X_test)
        class_report = classification_report(y_test, y_pred)
        print(f"Classification Report for {i}:\n", class_report)

        y_pred_val = rs.predict(X_val)
        class_report_val = classification_report(y_val, y_pred_val)
        print(f"Validation Classification Report for {i}:\n", class_report_val)

        try:
            y_pred_proba = rs.predict_proba(X_test)
            lroc.append(roc_auc_score(y_test, y_pred_proba, multi_class='ovr'))
        except AttributeError:
            print(f"{i} does not support probability prediction. ROC AUC score cannot be computed.")
            lroc.append(None)

        try:
            y_pred_proba_val = rs.predict_proba(X_val)
            lroc_val.append(roc_auc_score(y_val, y_pred_proba_val, multi_class='ovr'))
        except AttributeError:
            print(f"{i} does not support probability prediction. ROC AUC score cannot be computed for validation set.")
            lroc_val.append(None)

        end_time = time.time()
        training_time = end_time - start_time
        

        training_times.append(training_time)

        print(f"Completed training and evaluation for model {i} in {training_times[-1]:.2f} seconds at {time.ctime()}")
        print(f"Number of recorded training times: {len(training_times)}")

        lscore.append(accuracy_score(y_test, y_pred))
        lscore_val.append(accuracy_score(y_val, y_pred_val))  
        lcvscore.append(rs.best_score_) 
        
        from sklearn.exceptions import UndefinedMetricWarning
        import warnings

        try:
            lprecision.append(precision_score(y_test, y_pred, average='macro', zero_division=0))
            lprecision_val.append(precision_score(y_val, y_pred_val, average='macro', zero_division=0))
        except UndefinedMetricWarning:
            print(f"Warning encountered for model {i}")
            print("Test predictions:", y_pred)
            print("Validation predictions:", y_pred_val)

        
        lrecall.append(recall_score(y_test, y_pred, average='macro'))
        lrecall_val.append(recall_score(y_val, y_pred_val, average='macro'))  

        lfscore_macro.append(f1_score(y_test, y_pred, average='macro'))
        lfscore_macro_val.append(f1_score(y_val, y_pred_val, average='macro'))  

        lfscore_weighted.append(f1_score(y_test, y_pred, average='weighted'))
        lfscore_weighted_val.append(f1_score(y_val, y_pred_val, average='weighted'))  

        lrecall_severe.append(recall_score(y_test, y_pred, labels=[2], average='macro'))
        lrecall_severe_val.append(recall_score(y_val, y_pred_val, labels=[2], average='macro'))  

        lclass_report.append(class_report)
        lclass_report_val.append(class_report_val)  

        if lscore[-1] > max_score:
            max_score = lscore[-1]
            max_model = rs.best_estimator_
            max_model_params = rs.best_params_

        models_est.append(i)
        parameters.append(rs.best_params_)
        
    estimators_modelsearch['Models'] = models_est
    estimators_modelsearch['Best parameters'] = parameters
    estimators_modelsearch['ROC AUC'] = lroc
    estimators_modelsearch['ROC AUC Validation'] = lroc_val
    estimators_modelsearch['F1-score (macro)'] = lfscore_macro
    estimators_modelsearch['F1-score (macro) Validation'] = lfscore_macro_val
    estimators_modelsearch['F1-score (weighted)'] = lfscore_weighted
    estimators_modelsearch['F1-score (weighted) Validation'] = lfscore_weighted_val
    estimators_modelsearch['Precision'] = lprecision
    estimators_modelsearch['Precision Validation'] = lprecision_val
    estimators_modelsearch['Recall'] = lrecall
    estimators_modelsearch['Recall Severe'] = lrecall_severe
    estimators_modelsearch['Recall Severe Validation'] = lrecall_severe_val
    estimators_modelsearch['Score'] = lscore
    estimators_modelsearch['Score Validation'] = lscore_val  # Append validation scores
    estimators_modelsearch['CV Score'] = lcvscore
    print(len(estimators_modelsearch))
    print(len(processing_times))

    estimators_modelsearch['Processing Time'] = training_times

    plt.figure(figsize=(10,5))
    plt.bar(models_est, lscore, label='Test Score')
    plt.bar(models_est, lscore_val, label='Validation Score', alpha=0.5)
    plt.legend()
    plt.title("Performance Comparison")
    plt.show()

    return [max_score, max_model, max_model_params], estimators_modelsearch

In [ ]:
over = BorderlineSMOTE(sampling_strategy='auto')  
under = RandomUnderSampler(sampling_strategy='majority') 

results_proteomics, df_proteomics = model_search(
    models=models,
    params=params,
    over=over,
    under=under,
    X_train=X_proteomic_train_preprocessed,
    y_train=y_proteomic_train,
    X_test=X_proteomic_test_preprocessed,
    y_test=y_proteomic_test,
    X_val=X_proteomic_val_preprocessed,
    y_val=y_proteomic_val
)

In [ ]:
df_proteomics.to_excel('proteomics_model_training_results2.xlsx')

In [ ]:
df_proteomics

In [ ]:
over = BorderlineSMOTE(sampling_strategy='auto')  
under = RandomUnderSampler(sampling_strategy='majority')  

results_metabolomics, df_metabolomics = model_search(
    models=models,
    params=params,
    over=over,
    under=under,
    X_train=X_metabolomic_train_preprocessed,
    y_train=y_metabolomic_train,
    X_test=X_metabolomic_test_preprocessed,
    y_test=y_metabolomic_test,
    X_val=X_metabolomic_val_preprocessed,
    y_val=y_metabolomic_val
)

In [ ]:
df_metabolomics.to_excel('metabolomics_model_training_results2.xlsx')

In [ ]:
df_metabolomics

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

over = BorderlineSMOTE(sampling_strategy='auto')  
under = RandomUnderSampler(sampling_strategy='majority')

results_clinical, df_clinical = model_search(
    models=models,
    params=params,
    over=over,
    under=under,
    X_train=X_clinical_train,
    y_train=y_clinical_train,
    X_test=X_clinical_test,
    y_test=y_clinical_test,
    X_val=X_clinical_val,
    y_val=y_clinical_val
)

In [ ]:
df_clinical.to_excel('clinical_model_training_results2.xlsx')

In [ ]:
df_clinical

## Interpretability on Individual Models

In [ ]:
proteomics_model = xgb.XGBClassifier(
        subsample = 1.0,
        reg_lambda = 0.82, 
        reg_alpha = 0.8,
        n_estimators = 200,
        max_depth = 3,
        learning_rate = 0.25,
        gamma = 0.2,
        colsample_bytree = 0.8
)

metabolomics_model = SVC(
        probability = True,
        max_iter = 700, 
        kernel = 'rbf',
        gamma = 'scale',
        class_weight = 'balanced',
        C = 500,
)

#clinical_model = DecisionTreeClassifier(
        #splitter = 'best',
        #min_samples_split = 10,
        #min_samples_leaf = 2,
        #max_depth = 100,
        #criterion = 'entropy',
        #class_weight = 'balanced'
#) 

clinical_model = RandomForestClassifier(
    n_estimators = 50,
    min_samples_split = 10,
    min_samples_leaf =  2,
    max_depth = 20,
    criterion = 'entropy',
    class_weight = None
) 

In [ ]:
import os
import shap
import numpy as np
import pandas as pd
from lime import lime_tabular
import matplotlib.pyplot as plt

np.random.seed(42)

def train_and_explain_model(model, X_train, y_train, X_test, y_test, loading_matrix, explainer_type, shap_filename, lime_filename, cm_filename, output_filename, i):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    # Plotting the heatmap for confusion matrix
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.title('Confusion Matrix')

    # Save the figure for confusion matrix
    plt.savefig(cm_filename, dpi=300)
    plt.close()
    
    # SHAP Explainer
    if explainer_type == "tree":
        explainer = shap.TreeExplainer(model)
    elif explainer_type == "kernel":
        explainer = shap.KernelExplainer(model.predict_proba, X_train)
    elif explainer_type == "linear":
        explainer = shap.LinearExplainer(model, X_train)
    else:
        raise ValueError("Invalid explainer type")
    
    shap_values = explainer.shap_values(X_train)
    shap.summary_plot(shap_values, X_train, feature_names=X_train.columns, max_display=20, show=False)
    plt.savefig(shap_filename)
    plt.close()

    
    if isinstance(shap_values, list):  
        vals = np.abs(shap_values[0]).mean(0)
    else:
        vals = np.abs(shap_values).mean(0)

    feature_importance = pd.DataFrame(list(zip(X_train.columns, vals)), columns=['col_name', 'feature_importance_vals'])
    feature_importance.sort_values(by=['feature_importance_vals'], ascending=False, inplace=True)
    feature_importance.to_excel(output_filename)

    # Loadings
    if loading_matrix is not None:
        assert isinstance(loading_matrix, pd.DataFrame), "loading_matrix should be a pandas DataFrame."
        top_features = [i for i in list(feature_importance['col_name'])[:30] if str(i).isdigit()]
        print(top_features)
        loading_trans = loading_matrix[top_features].T
        loading_pow = {}
        for column in loading_trans.columns:
            magnitude = loading_trans[column].pow(2).sum()
            loading_pow[column] = magnitude
        pd.DataFrame.from_dict(loading_pow, orient='index').to_excel(output_filename.replace('.xlsx', '_loading_pow.xlsx'))

    # LIME Explainer
    explainer = lime_tabular.LimeTabularExplainer(X_train.to_numpy(), feature_names=X_train.columns, class_names=["Mild", "Moderate", "Severe"], mode='classification', random_state=42)
    exp = explainer.explain_instance(X_test.iloc[i].to_numpy(), model.predict_proba, num_features=10)
    exp.save_to_file(os.path.join(os.getcwd(), lime_filename))
    
    observation = X_test.iloc[i]
    observation_df = pd.DataFrame(observation).transpose()
    observation_df['who_severity'] = y_test.iloc[i]
    print(observation_df)
    
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba


In [ ]:
clinical_y_pred_proba = train_and_explain_model(clinical_model, X_clinical_train, y_clinical_train, X_clinical_test, y_clinical_test, None, 'tree', 'clin_shap.svg', 'clin_lime.html', 'clin_confusion_matrix.png', 'clin_feature_importance.xlsx', 9)

proteomics_y_pred_proba = train_and_explain_model(proteomics_model, X_proteomic_train_preprocessed, y_proteomic_train, X_proteomic_test_preprocessed, y_proteomic_test, loading_matrix_proteomics, 'tree', 'pro_shap.svg', 'pro_lime.html', 'pro_confusion_matrix.png', 'pro_feature_importance.xlsx', 10)

metabolomics_y_pred_proba = train_and_explain_model(metabolomics_model, X_metabolomic_train_preprocessed, y_metabolomic_train, X_metabolomic_test_preprocessed, y_metabolomic_test, loading_matrix_metabolomics, 'kernel', 'met_shap.svg', 'met_lime.html', 'met_confusion_matrix.png', 'met_feature_importance.xlsx', 23)

Examine the Feature Importance Output

In [ ]:
feature_names = X_proteomic_train_preprocessed.columns

instance_index = 10

observation = X_proteomic_test_preprocessed.iloc[instance_index]

observation_df = pd.DataFrame(observation).transpose()
observation_df['who_severity'] = y_proteomic_test.iloc[instance_index]
print(observation_df)

In [ ]:
feature_names = X_metabolomic_train_preprocessed.columns

instance_index = 9

observation = X_metabolomic_test_preprocessed.iloc[instance_index]

observation_df = pd.DataFrame(observation).transpose()
observation_df['who_severity'] = y_metabolomic_test.iloc[instance_index]
print(observation_df)

In [ ]:
feature_names = X_clinical_train.columns

instance_index = 23

observation = X_clinical_test.iloc[instance_index]

observation_df = pd.DataFrame(observation).transpose()
observation_df['who_severity'] = y_clinical_test.iloc[instance_index]
print(observation_df)

## Stacking

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class CustomModel(BaseEstimator, ClassifierMixin):
    def __init__(self, model, feature_indices):
        self.model = model
        self.feature_indices = feature_indices

    def fit(self, X, y=None):
        self.model.fit(X[:, self.feature_indices], y)
        self.classes_ = self.model.classes_  # Set the classes_ attribute
        return self
    
    def predict_proba(self, X):
        return self.model.predict_proba(X[:, self.feature_indices])

    def predict(self, X):
        return self.model.predict(X[:, self.feature_indices])

    def get_params(self, deep=True):
        return {"model": self.model, "feature_indices": self.feature_indices}

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)


In [ ]:
X_train = np.concatenate([X_metabolomic_train_preprocessed, X_proteomic_train_preprocessed, X_clinical_train], axis=1)
X_test = np.concatenate([X_metabolomic_test_preprocessed, X_proteomic_test_preprocessed, X_clinical_test], axis=1)
X_val = np.concatenate([X_metabolomic_val_preprocessed, X_proteomic_val_preprocessed, X_clinical_val], axis=1)
y_train = y_clinical_train
y_test = y_clinical_test
y_val = y_clinical_val

metabolomics_feature_indices = list(range(X_metabolomic_train_preprocessed.shape[1]))
proteomics_feature_indices = list(range(X_metabolomic_train_preprocessed.shape[1], X_metabolomic_train_preprocessed.shape[1] + X_proteomic_train_preprocessed.shape[1]))
clinical_feature_indices = list(range(X_metabolomic_train_preprocessed.shape[1] + X_proteomic_train_preprocessed.shape[1], X_train.shape[1]))

In [ ]:
X_metabolomic_train_preprocessed.shape

In [ ]:
X_proteomic_train_preprocessed.shape

In [ ]:
X_clinical_train.shape

In [ ]:
X_train.shape

In [ ]:
proteomics_model

In [ ]:
metabolomics_model

In [ ]:
clinical_model

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_score

base_models = [
    ('svc', CustomModel(metabolomics_model, metabolomics_feature_indices)),
    ('xgb', CustomModel(proteomics_model, proteomics_feature_indices)),
    ('rf', CustomModel(clinical_model, clinical_feature_indices)),
]

stacked_models = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(multi_class='multinomial', penalty = 'l2', random_state=42))

print(f"Starting model training at: {time.ctime()}")
start_time = time.time()  

stacked_models.fit(X_train, y_train)

y_pred = stacked_models.predict(X_test)
y_pred_proba = stacked_models.predict_proba(X_test)
y_pred_val = stacked_models.predict(X_val)
y_pred_proba_val = stacked_models.predict_proba(X_val)

score = accuracy_score(y_test, y_pred)
score_val = accuracy_score(y_val, y_pred_val)
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
roc_auc_val = roc_auc_score(y_val, y_pred_proba_val, multi_class='ovr')
fscore_macro = f1_score(y_test, y_pred, average='macro')
fscore_macro_val = f1_score(y_val, y_pred_val, average='macro')
fscore_weighted = f1_score(y_test, y_pred, average='weighted')
fscore_weighted_val = f1_score(y_val, y_pred_val, average='weighted')
precision = precision_score(y_test, y_pred, average='macro')
precision_val = precision_score(y_val, y_pred_val, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
recall_val = recall_score(y_val, y_pred_val, average='macro')
recall_severe = recall_score(y_test, y_pred, labels=[2], average='macro')
recall_severe_val = recall_score(y_val, y_pred_val, labels=[2], average='macro')

        
end_time = time.time()
processing_time = end_time - start_time
        
X_full = np.concatenate([X_train, X_test, X_val], axis=0)
y_full = np.concatenate([y_train, y_test, y_val], axis=0)

cvscore = cross_val_score(stacked_models, X_full, y_full, cv=StratifiedKFold(10), scoring='roc_auc_ovr').mean()

estimators_model = pd.DataFrame()
estimators_model['Models'] = ['Stacking Ensemble Model']
estimators_model['ROC AUC'] = [roc_auc]
estimators_model['ROC AUC Validation'] = [roc_auc_val]
estimators_model['F1-score (macro)'] = [fscore_macro]
estimators_model['F1-score (macro) Validation'] = [fscore_macro_val]
estimators_model['F1-score (weighted)'] = [fscore_weighted]
estimators_model['F1-score (weighted) Validation'] = [fscore_weighted_val]
estimators_model['Precision'] = [precision]
estimators_model['Precision Validation'] = [precision_val]
estimators_model['Recall'] = [recall]
estimators_model['Recall Validation'] = [recall_val]
estimators_model['Recall Severe'] = [recall_severe]
estimators_model['Recall Severe Validation'] = [recall_severe_val]
estimators_model['Score'] = [score]
estimators_model['Score Validation'] = [score_val]  
estimators_model['CV Score'] = [cvscore]
estimators_model['Training Time'] = [processing_time]

In [ ]:
estimators_model.to_excel('stacked_model_results.xlsx')

In [ ]:
print(estimators_model)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.savefig('confusion_matrix.png', dpi=300)  
plt.show()